##### Current setup

In [ ]:
import os
import re
import sys
import json
import math
import traceback
from pathlib import Path

import numpy as np
import soundfile as sf
from praatio import tgio

REF_DIR = Path("/Users/moanason/Downloads/Data_TEMP")  # contains rec_sXX_NC#.wav
REF_RX  = re.compile(r"^rec_s(?P<sess>\d+)_NC(?P<nc>\d+)\.wav$", re.IGNORECASE)

# tier names to search for. We'll auto-detect, but these are preferred names.
PREFERRED_A_TIER_NAMES = ["Diarisation_A", "Diarization_A", "Diarisation-A", "A", "SpeakerA"]
PREFERRED_B_TIER_NAMES = ["Diarisation_B", "Diarization_B", "Diarisation-B", "B", "SpeakerB"]

# New tier we will (re)create in the reference TextGrid, uncomment to change name.
# TRANS_TIER_NAME = "Trans_Condition"

# knobs
MERGE_TOL   = 0.002    # merge if gap/overlap <= 2 ms
MIN_SEG_DUR = 0.002     # drop segments shorter than 20 ms
WARN_TOL    = 0.001    # warn if |TG xmax - WAV dur| > 1 ms


def open_textgrid_compat(path_str):
    """
    Open a TextGrid with praatio across versions.
    Some versions accept no kwargs; others support includeEmptyIntervals, reportingMode.
    """
    try:
        return tgio.openTextgrid(path_str)
    except TypeError:
        # fallback older signature that *might* accept includeEmptyIntervals
        try:
            return tgio.openTextgrid(path_str, includeEmptyIntervals=True)
        except TypeError:
            # final fallback: no kwargs again (just raise the original)
            return tgio.openTextgrid(path_str)


def read_duration_sec(path_wav):
    with sf.SoundFile(str(path_wav)) as f:
        return len(f) / float(f.samplerate)


def clip_and_merge(intervals, total_dur, min_len=0.0, tol=0.0):
    """Clip to [0, total_dur], drop very short, and merge close/overlapping."""
    if not intervals:
        return []
    # clip
    clipped = []
    for s, e in intervals:
        s = max(0.0, float(s))
        e = min(float(total_dur), float(e))
        if e - s >= min_len:
            clipped.append((s, e))
    if not clipped:
        return []

    clipped.sort()
    merged = []
    s0, e0 = clipped[0]
    for s, e in clipped[1:]:
        if s <= e0 + tol:
            e0 = max(e0, e)
        else:
            merged.append((s0, e0))
            s0, e0 = s, e
    merged.append((s0, e0))
    return merged


def intervals_from_tier(tier):
    out = []
    for (s, e, lab) in tier.entryList:
        if lab and str(lab).strip():
            out.append((float(s), float(e)))
    return out


def find_diar_tiers(tg, session_code):
    tier_names = tg.tierNameList

    # 1) Try preferred names
    a_name = next((n for n in PREFERRED_A_TIER_NAMES if n in tier_names), None)
    b_name = next((n for n in PREFERRED_B_TIER_NAMES if n in tier_names), None)
    if a_name and b_name:
        return a_name, b_name

    # 2) Detect by labels matching sXX_A / sXX_B
    pat_A = re.compile(r"^s0*%s_A$" % session_code, re.IGNORECASE)
    pat_B = re.compile(r"^s0*%s_B$" % session_code, re.IGNORECASE)

    a_guess = None
    b_guess = None
    for name in tier_names:
        tier = tg.tierDict[name]
        if not isinstance(tier, tgio.IntervalTier):
            continue
        labels = {str(lab).strip() for (_, _, lab) in tier.entryList if str(lab).strip()}
        if any(pat_A.match(lbl) for lbl in labels):
            a_guess = a_guess or name
        if any(pat_B.match(lbl) for lbl in labels):
            b_guess = b_guess or name
        if a_guess and b_guess:
            break

    return a_guess, b_guess


def build_trans_condition(a_intervals, b_intervals, total_dur, label_turn_a, label_turn_b):
    """
    Given merged speech intervals for A and B, produce a piecewise-constant segmentation:
      - Silence (none active)
      - Turn_sXX_A (only A active)
      - Turn_sXX_B (only B active)
      - Overlap (both active)
    Returns merged list of (start, end, label) cleaned with tolerances.
    """
    # boundaries from both speakers + [0, total_dur]
    bounds = [0.0, total_dur]
    for s, e in a_intervals + b_intervals:
        bounds.append(max(0.0, min(total_dur, s)))
        bounds.append(max(0.0, min(total_dur, e)))
    bounds = sorted(set(bounds))

    # sweep-line: pointers into a_intervals / b_intervals
    def active_in(span_start, span_end, intervals):
        # consider active if at least 50% of span overlaps any interval
        span_len = span_end - span_start
        if span_len <= 0:
            return False
        for (s, e) in intervals:
            ov = max(0.0, min(span_end, e) - max(span_start, s))
            if ov >= 0.5 * span_len:
                return True
        return False

    segs = []
    for i in range(len(bounds) - 1):
        s = bounds[i]
        e = bounds[i + 1]
        if e - s <= 0:
            continue
        a_on = active_in(s, e, a_intervals)
        b_on = active_in(s, e, b_intervals)
        if a_on and b_on:
            lab = "Overlap"
        elif a_on and not b_on:
            lab = label_turn_a
        elif b_on and not a_on:
            lab = label_turn_b
        else:
            lab = "Silence"
        segs.append((s, e, lab))

    # merge adjacent same-label segments
    merged = []
    if segs:
        s0, e0, l0 = segs[0]
        for s, e, lab in segs[1:]:
            if lab == l0 and s <= e0 + MERGE_TOL:
                e0 = max(e0, e)
            else:
                if e0 - s0 >= MIN_SEG_DUR:
                    merged.append((s0, e0, l0))
                s0, e0, l0 = s, e, lab
        if e0 - s0 >= MIN_SEG_DUR:
            merged.append((s0, e0, l0))

    # clip strictly and final small cleanup
    cleaned = []
    for s, e, l in merged:
        s = max(0.0, s)
        e = min(total_dur, e)
        if e - s >= MIN_SEG_DUR:
            cleaned.append((s, e, l))

    return cleaned


def save_textgrid_for(path_wav, tg, output_root):
    if output_root is None:
        out_path = Path(path_wav).with_suffix(".TextGrid")
    else:
        rel = Path(path_wav).name.replace(".wav", ".TextGrid")
        out_path = Path(output_root) / rel
        out_path.parent.mkdir(parents=True, exist_ok=True)

    tg.save(str(out_path), minimumIntervalLength=0.0, outputFormat="textgrid")
    return out_path


def main():
    ref_wavs = sorted([p for p in REF_DIR.glob("rec_s*_NC*.wav") if p.is_file()])
    print(f"Found {len(ref_wavs)} reference WAVs in {REF_DIR}")

    updated = 0
    errors = []

    for wav_path in ref_wavs:
        try:
            m = REF_RX.match(wav_path.name)
            if not m:
                continue
            sess = m.group("sess")  # e.g., '03'
            nc   = m.group("nc")    # e.g., '1'

            tg_path = wav_path.with_suffix(".TextGrid")
            if not tg_path.exists():
                raise RuntimeError("Missing tg next to reference WAV. "
                                   "Run the diarisation-to-reference step first.")

            tg = open_textgrid_compat(str(tg_path))
            total_dur = read_duration_sec(wav_path)

            a_name, b_name = find_diar_tiers(tg, sess)
            if not a_name or not b_name:
                raise RuntimeError(f"Could not locate diarisation tiers for session s{sess}: "
                                   f"A={a_name}, B={b_name}; available={tg.tierNameList}")

            tierA = tg.tierDict[a_name]
            tierB = tg.tierDict[b_name]
            if not isinstance(tierA, tgio.IntervalTier) or not isinstance(tierB, tgio.IntervalTier):
                raise RuntimeError("Diarisation tiers are not IntervalTier.")

            A_intervals = intervals_from_tier(tierA)
            B_intervals = intervals_from_tier(tierB)
            A_intervals = clip_and_merge(A_intervals, total_dur, min_len=MIN_SEG_DUR, tol=MERGE_TOL)
            B_intervals = clip_and_merge(B_intervals, total_dur, min_len=MIN_SEG_DUR, tol=MERGE_TOL)

            # build labels Turn_sXX_A / Turn_sXX_B
            label_turn_a = f"Turn_s{int(sess):02d}_A"
            label_turn_b = f"Turn_s{int(sess):02d}_B"

            trans_entryList = build_trans_condition(
                A_intervals, B_intervals, total_dur,
                label_turn_a=label_turn_a,
                label_turn_b=label_turn_b
            )

            if TRANS_TIER_NAME in tg.tierNameList: # remove existing
                tg.removeTier(TRANS_TIER_NAME)

            trans_tier = tgio.IntervalTier(TRANS_TIER_NAME, trans_entryList, 0.0, float(total_dur))
            tg.addTier(trans_tier)

            save_textgrid_for(wav_path, tg, output_root=None)
            updated += 1

            tg_xmax = float(total_dur)  # by construction
            if abs(tg_xmax - total_dur) > WARN_TOL:
                print(f"[WARN] TG xmax != WAV duration for {wav_path.name}: TG={tg_xmax:.6f}s WAV={total_dur:.6f}s")

        except Exception as e:
            errors.append((str(wav_path), traceback.format_exc()))

    print(f"Trans_Condition from diarisation: updated {updated} TextGrids; errors={len(errors)}.")
    if errors:
        print("Errors (first 10):")
        for p, err in errors[:10]:
            line = err.strip().splitlines()[-1]
            print(f"  - {p} : {line}")


if __name__ == "__main__":
    main()

Found 0 reference WAVs in /Users/moanason/Downloads/Data_TEMP
Trans_Condition from diarisation: updated 0 TextGrids; errors=0.


##### For any other versions of praatio or different annotation preferences, use this.

In [ ]:
import re
import csv
import math
import traceback
from pathlib import Path

import numpy as np
import soundfile as sf
from praatio import tgio

REF_DIR = Path("/Users/moanason/Downloads/Data_TEMP")   # contains ref/rec_sXX_NC#.wav + TextGrids
WAV_RX  = re.compile(r"^ref_(s\d{2})_NC(\d+)_processed\.wav$", re.IGNORECASE)

# TRANS_TIER_NAME = "TransCondition"
# TRANS_TIER_NAME = "TransEvents" # adding the same tier for manual processing

DIAR_TIER_A_CANDIDATES = [
    "Diarisation_A", "Diarization_A", "Diarisation A", "Diarization A",
    "A", "SpkA", "SpeakerA"
]
DIAR_TIER_B_CANDIDATES = [
    "Diarisation_B", "Diarization_B", "Diarisation B", "Diarization B",
    "B", "SpkB", "SpeakerB"
]

# same knobs
MIN_SEG_DUR = 0.002   # drop segments shorter than 2 ms after clipping
MERGE_TOL   = 0.002  # merge if gap/overlap <= 2 ms
WARN_TOL    = 0.001  # warn if |TG xmax - WAV dur| > 1 ms



def read_duration_sec(path):
    with sf.SoundFile(str(path)) as f:
        return len(f) / float(f.samplerate)

def open_textgrid_safe(tg_path):
    return tgio.openTextgrid(str(tg_path))

def get_tier_entries(tier):
    if hasattr(tier, "entries"):
        return tier.entries
    elif hasattr(tier, "entryList"):
        return tier.entryList
    else:
        raise AttributeError("Unknown Praatio IntervalTier entries attribute")

def has_tier(tg, name_lower):
    names = [n.lower() for n in tg.tierNameList] if hasattr(tg, "tierNameList") else [n.lower() for n in tg.tierDict.keys()]
    return name_lower in names

def remove_tier_if_exists(tg, name):
    try:
        if has_tier(tg, name.lower()):
            tg.removeTier(name)
    except Exception:
        pass

def find_tier_by_name_candidates(tg, candidates):
    lower_map = {}
    names = tg.tierNameList if hasattr(tg, "tierNameList") else list(tg.tierDict.keys())
    for n in names:
        lower_map[n.lower()] = n
    for cand in candidates:
        n = lower_map.get(cand.lower())
        if n is not None:
            tier = tg.tierDict[n] if hasattr(tg, "tierDict") else tg.tierDict[n]  # both exist in modern praatio
            if isinstance(tier, tgio.IntervalTier):
                return n, tier
    return None, None

def any_label_matches(entries, target_label):
    for (s, e, lab) in entries:
        if str(lab).strip().lower() == target_label.strip().lower():
            return True
    return False

def find_tier_for_label_suffix(tg, label_suffix):
    # look for a tier that contains labels like "..._A" or "..._B"
    names = tg.tierNameList if hasattr(tg, "tierNameList") else list(tg.tierDict.keys())
    for n in names:
        tier = tg.tierDict[n]
        if not isinstance(tier, tgio.IntervalTier):
            continue
        entries = get_tier_entries(tier)
        for (_, _, lab) in entries:
            lab_s = str(lab).strip().lower()
            if lab_s.endswith(label_suffix.lower()):
                return n, tier
    return None, None

def clip_intervals(intervals, total_dur, min_len=0.0):
    out = []
    for s, e in intervals:
        s = max(0.0, float(s))
        e = min(float(total_dur), float(e))
        if e - s >= min_len:
            out.append((s, e))
    return out

def merge_intervals(intervals, tol=0.0):
    if not intervals:
        return []
    intervals = sorted(intervals)
    merged = []
    s0, e0 = intervals[0]
    for s, e in intervals[1:]:
        if s <= e0 + tol:
            e0 = max(e0, e)
        else:
            merged.append((s0, e0))
            s0, e0 = s, e
    merged.append((s0, e0))
    return merged

def collapse_same_label(intervals_labeled, tol=0.0):
    if not intervals_labeled:
        return []
    out = []
    (s0, e0, lab0) = intervals_labeled[0]
    for (s, e, lab) in intervals_labeled[1:]:
        if lab == lab0 and s <= e0 + tol:
            e0 = max(e0, e)
        else:
            out.append((s0, e0, lab0))
            s0, e0, lab0 = s, e, lab
    out.append((s0, e0, lab0))
    return out

def intervals_from_tier(tier, total_dur):
    raw = get_tier_entries(tier)
    segs = [(float(s), float(e)) for (s, e, _lab) in raw]
    segs = clip_intervals(segs, total_dur, min_len=MIN_SEG_DUR)
    segs = merge_intervals(segs, tol=MERGE_TOL)
    return segs

def build_trans_condition(segs_A, segs_B, total_dur, sess_code):
    """
    segs_A, segs_B: lists of (s,e) "speech" for A/B
    Return non-overlapping list of (s,e,label) with labels in {Silence, Turn_sXX_A, Turn_sXX_B, Overlap}.
    """
    # collect breakpoints
    points = {0.0, float(total_dur)}
    for s, e in segs_A: points.add(s); points.add(e)
    for s, e in segs_B: points.add(s); points.add(e)
    bps = sorted(points)
    def active(mid, segs):
        for s, e in segs:
            if s <= mid < e:
                return True
        return False

    labeled = []
    for i in range(len(bps) - 1):
        s = bps[i]; e = bps[i+1]
        if e - s < MIN_SEG_DUR:  # drop micro-fragments
            continue
        mid = 0.5 * (s + e)
        A_on = active(mid, segs_A)
        B_on = active(mid, segs_B)
        if A_on and B_on:
            lab = "Overlap"
        elif A_on:
            lab = f"Turn_{sess_code}_A"
        elif B_on:
            lab = f"Turn_{sess_code}_B"
        else:
            lab = "Silence"
        labeled.append((s, e, lab))

    # merge adjacent same-label
    labeled = collapse_same_label(labeled, tol=MERGE_TOL)
    # final clip/clean
    labeled = [(max(0.0, s), min(total_dur, e), lab) for (s, e, lab) in labeled if (e - s) >= MIN_SEG_DUR]
    return labeled

def save_textgrid(path_wav, tg, output_root):
    total_dur = read_duration_sec(path_wav)
    if output_root is None:
        out_path = path_wav.with_suffix(".TextGrid")
    else:
        rel = path_wav.relative_to(REF_DIR)
        out_path = (output_root / rel).with_suffix(".TextGrid")
        out_path.parent.mkdir(parents=True, exist_ok=True)
    tg.save(str(out_path), minimumIntervalLength=0.0, outputFormat="textgrid")
    # sanity note about xmax
    if abs(total_dur - total_dur) > WARN_TOL:  # constructed to match exactly
        print(f"[WARN] TG xmax != WAV duration for {path_wav.name}")
    return out_path

# main fuction
def main():
    ref_wavs = sorted([p for p in REF_DIR.glob("ref_s*_NC*_processed.wav") if p.is_file()])
    print(f"Found {len(ref_wavs)} reference WAVs in {REF_DIR}")

    updated = 0
    errors = []

    for ref_wav in ref_wavs:
        m = WAV_RX.match(ref_wav.name)
        if not m:
            continue
        sess_code = m.group(1)   # e.g., s03
        # nc = m.group(2)        # not needed for labels (but available)
        tg_path = ref_wav.with_suffix(".TextGrid")
        if not tg_path.exists():
            errors.append((str(ref_wav), "RuntimeError: Missing tg next to ref WAV. Run the diarisation-to-reference step first."))
            continue

        try:
            tg = open_textgrid_safe(tg_path)
            total_dur = read_duration_sec(ref_wav)

            nameA, tierA = find_tier_by_name_candidates(tg, DIAR_TIER_A_CANDIDATES)
            if tierA is None:
                nameA, tierA = find_tier_for_label_suffix(tg, f"{sess_code}_A")
            nameB, tierB = find_tier_by_name_candidates(tg, DIAR_TIER_B_CANDIDATES)
            if tierB is None:
                nameB, tierB = find_tier_for_label_suffix(tg, f"{sess_code}_B")

            if tierA is None or tierB is None:
                raise RuntimeError(f"Could not find diarisation tiers for A/B in {tg_path.name}. "
                                   f"FoundA={nameA is not None}, FoundB={nameB is not None}")

            segs_A = intervals_from_tier(tierA, total_dur)
            segs_B = intervals_from_tier(tierB, total_dur)

            trans_entries = build_trans_condition(segs_A, segs_B, total_dur, sess_code)

            remove_tier_if_exists(tg, TRANS_TIER_NAME)
            trans_tier = tgio.IntervalTier(TRANS_TIER_NAME, trans_entries, 0.0, float(total_dur))
            tg.addTier(trans_tier)

            save_textgrid(ref_wav, tg, output_root=None)
            updated += 1

        except Exception as e:
            errors.append((str(ref_wav), f"{type(e).__name__}: {e}"))

    print(f"Trans_Condition from diarisation: updated {updated} TextGrids; errors={len(errors)}.")
    if errors:
        print("Errors (first 10):")
        for p, msg in errors[:10]:
            print(f"  - {p} : {msg}")

if __name__ == "__main__":
    main()


Found 10 reference WAVs in /Users/moanason/Downloads/Data_TEMP
Trans_Condition from diarisation: updated 10 TextGrids; errors=0.
